# SoccerNet GSR — Official Baseline Test

This notebook runs the **unchanged** SoccerNet Game State Reconstruction baseline
on official SoccerNet validation data. The goal is to prove the environment works
before introducing custom video clips.

**Requirements:** Google Colab with GPU runtime
(Runtime → Change runtime type → T4 GPU)

In [ ]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA version: {torch.version.cuda}")

assert torch.cuda.is_available(), "No GPU detected! Go to Runtime → Change runtime type → T4 GPU"

## Setup — Clone repos & install dependencies

In [ ]:
import os

# Clone our repo
REPO_URL = "https://github.com/Moiz005/SoccerVision-Player-Tracking-3D-Reconstruction.git"
REPO_NAME = "SoccerVision-Player-Tracking-3D-Reconstruction"

if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}

%cd {REPO_NAME}

# Clone sn-gamestate and tracklab
if not os.path.exists("sn-gamestate"):
    !git clone https://github.com/SoccerNet/sn-gamestate.git

if not os.path.exists("tracklab"):
    !git clone https://github.com/TrackingLaboratory/tracklab.git

# Install sn-gamestate (this installs tracklab and all deps)
%cd sn-gamestate
!pip install -e .

# Install mmcv via mim (must NOT use pip directly)
!pip install mim
!mim install mmcv==2.0.1

In [ ]:
import tracklab
import sn_gamestate

print(f"TrackLab version: {tracklab.__version__}")
print(f"SN-GameState imported successfully")
print("All dependencies OK")

## Download Sample Data

The sn-gamestate pipeline **auto-downloads** the SoccerNet validation dataset
and pretrained model weights on first run. This may take a few minutes.

No manual download needed — just run the cell below.

## Run Baseline on Official Validation Video

This runs the full GSR pipeline on **1 validation video** from SoccerNet:
- Player detection (YOLOv11)
- Re-ID & tracking (PRTReid + StrongSORT)
- Pitch localization (NBW calibration)
- Jersey number detection (MMOCR)
- Team assignment (K-means)
- Game state reconstruction

The first run will download ~35GB of dataset and model weights.

In [ ]:
%cd /content/{REPO_NAME}/sn-gamestate

# Run the official SoccerNet GSR baseline
# This processes 1 validation video (nvid: 1) by default
# First run downloads dataset + weights automatically
!tracklab -cn soccernet

## View Results

The pipeline generates:
- Annotated visualization video (bounding boxes, IDs, pitch overlay)
- Tracker state file (.pklz)
- Evaluation metrics (GS-HOTA)

In [ ]:
import glob
from IPython.display import Video, display

# Find generated visualization videos
output_videos = glob.glob(
    "output/**/visualization/videos/*.mp4",
    recursive=True
)

print(f"Found {len(output_videos)} output video(s):")
for v in output_videos:
    print(f"  - {v}")

if output_videos:
    print(f"\nPlaying: {output_videos[0]}")
    display(Video(output_videos[0], width=800))
else:
    print("No output videos found. Check the pipeline output above for errors.")

## Next Steps

If this ran successfully, the baseline works on official data.

**Next:** Run this on your own football clip → Phase 4 (Custom Video Adapter)